# az-sentiment: Colab run-ları
**Runtime → Change runtime type → T4 GPU.** Hücrələri ardıcıl işlət, tabı açıq saxla (Colab boş qalanda bağlanır).

Repoda artıq var: təmiz split (`data/processed/`), baseline, TF-IDF, çəkili XLM-R nəticələri (`results/`).
Burada qalan 3 run işləyir:
1. XLM-R **çəkisiz** (çəkili ilə müqayisə üçün) → arxa planda, ~50–60 dəq
2. deepseek-r1:1.5b **0-shot**, 500 nümunə → ~30–40 dəq
3. deepseek-r1:1.5b **3-shot**, 500 nümunə → ~30–40 dəq

1 və 2–3 paralel gedir (T4-də 15 GB VRAM bəs edir). Cəmi **~1–1.5 saat**.

## 1. Kod + asılılıqlar

In [2]:
!git clone -q https://github.com/fuadsadiqov/az-sentiment.git /content/az-sentiment
%cd /content/az-sentiment
!pip install -q -e . tabulate
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
!ls data/processed results/metrics

fatal: destination path '/content/az-sentiment' already exists and is not an empty directory.
/content/az-sentiment
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for az-sentiment (pyproject.toml) ... done
2.11.0+cu130 True Tesla T4
data/processed:
naive_test.parquet   test.parquet   val.parquet
naive_train.parquet  train.parquet

results/metrics:
baseline_majority.json	   tfidf_logreg.json
baseline_random.json	   tfidf_logreg_NAIVE_split.json
tfidf_logreg_cw-None.json  xlm-roberta-base_weighted.json


## 2. XLM-R çəkisiz: arxa planda başlat
Log: `results/log_xlmr_plain.txt`. T4-də skript avtomatik fp16 istifadə edir.

In [ ]:
!nohup python scripts/04_transformer.py > results/log_xlmr_plain.txt 2>&1 &

## 3. Ollama + deepseek-r1:1.5b

In [4]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull deepseek-r1:1.5b

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



*`italicized text`*## 4. LLM run-ları (XLM-R arxada işləyərkən)

In [5]:
!python scripts/05_llm.py --shots 0 --n 500

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (

[llm_deepseek-r1-1.5b_0shot] acc=0.510  macro_f1=0.368  neg: P=0.33 R=0.65  neu: P=0.03 R=0.14  pos: P=0.85 R=0.48


In [ ]:
!python scripts/05_llm.py --shots 3 --n 500

## 5. XLM-R-in bitməsini yoxla

> Add blockquote



```
`# This is formatted as code`
```


Son sətirdə `[xlm-roberta-base] acc=... macro_f1=...` görünəndə bitib. Görünmürsə bir neçə dəqiqə sonra yenidən işlət.

In [6]:
!tail -c 400 results/log_xlmr_plain.txt | tr '
' '
' | tail -3
!ls results/metrics

SyntaxError: unterminated string literal (detected at line 3) (1679202636.py, line 3)

## 6. Müqayisə + səhv təhlili

In [ ]:
!python scripts/06_compare.py
!python scripts/07_error_analysis.py --model xlm-roberta-base_weighted --other tfidf_logreg

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/comparison.md", encoding="utf-8").read()))
display(Image("results/figures/comparison.png"))

## 7. Nəticələri endir
`results.zip`-i lokal repoya aç (`az-sentiment/` qovluğunda unzip, `results/` üstünə yazılır).

In [ ]:
!zip -q -r results.zip results -x "results/log_*"
from google.colab import files
files.download("results.zip")